# 05 — Why Post-Hoc Correction Cannot Work (Table 5)

The obvious response to a script-induced score gap is to correct it away: shift
the romanised scores, rescale them, or map their distribution onto the native
one. This notebook shows why none of that helps where it matters.

**The impossibility.** Spearman ρ is invariant under any strictly monotone
transform. Every corrector below — mean shift, affine rescaling, quantile
mapping, isotonic regression — is monotone within a language. So all of them
close the *gap*, and none of them recovers *within-language ranking*: `ρ_lang` is
frozen across the entire table while the native ceiling sits far above it.

Correcting the location of the scores cannot restore information the encoder
never encoded.

**Base:** 6,995, split 50/50 within each language (`RandomState(42).choice`, see Step 2).
All correctors are fitted on the train half and evaluated on the test half.

**Input:** `../data/indic/indic_parity_xlmr.xlsx`
**Output:** `../results/tables/table5_corrector_impossibility.csv`

> These numbers depend on the split RNG. They are registered with status
> `script` in `paper_numbers.yaml` and are reproducible only under seed 42.

## Step 0 — Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# ── Paths (relative; nothing in this repository uses an absolute path) ───────
DATA_XLMR   = Path("../data/indic/indic_parity_xlmr.xlsx")
DATA_MULTI  = Path("../data/indic/indic_parity_multi_tokenizer.xlsx")
DATA_LATIN  = Path("../data/latin/wmt24_ende_enes_metrics.xlsx")
TABLES_DIR  = Path("../results/tables")
FIGURES_DIR = Path("../results/figures")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Sheet names exactly as they appear in the workbook ───────────────────────
SHEET_MAP = {
    "GUJ": "Indic_mt _for_analysis - Gujara",
    "TAM": "Indic_mt _for_analysis - Tamil_",
    "MAL": "Indic_mt _for_analysis - Malaya",
    "MAR": "Indic_mt _for_analysis - Marath",
    "HIN": "Indic_mt _for_analysis - Hindi_",
}

# ── Language display order (fixed throughout the paper) ──────────────────────
LANG_ORDER = ["GUJ", "TAM", "MAL", "MAR", "HIN"]

# ── Column names (as in the xlsx) ────────────────────────────────────────────
COL_COMET_NAT = "COMET"                                          # native-script COMET
COL_COMET_ROM = "COMET_romanized"                                # romanised COMET
COL_TP_NAT    = "Translation_xlmr_TP"                            # TP, native
COL_TP_ROM    = "Translation_Transliteration_romanized_xlmr_TP"  # TP, romanised
COL_IP_NAT    = "Translation_xlmr_IP"                            # IP, native
COL_IP_ROM    = "Translation_Transliteration_romanized_xlmr_IP"  # IP, romanised
COL_HUMAN     = "Human_scores"                                   # MQM-derived human score
COL_SEVERITY  = "Error1_Severity"                                # primary error severity

# ── Seeds (every stochastic step in this repository) ─────────────────────────
SEED_SPLIT = 42   # 50/50 within-language train/test split
SEED_GBM   = 0    # GradientBoostingRegressor
SEED_PERM  = 0    # paired permutation test

print("Config loaded. DATA_XLMR:", DATA_XLMR)

## Step 1 — Load and Assemble the Working Set

In [ ]:
def load_sheets(path):
    """Read the five per-language sheets.

    Returns two dicts keyed by ISO code:
      full  — all 1,400 rows per language (the 7,000-segment base)
      work  — rows carrying a numeric human score (the 6,995-segment base)

    Coercing the human-score column to numeric is what removes the five
    unusable rows: four are blank and one (Malayalam) holds the string
    ``\`19``, which is not a score.
    """
    full, work = {}, {}
    for lang in LANG_ORDER:
        d = pd.read_excel(path, sheet_name=SHEET_MAP[lang])
        d["H"] = pd.to_numeric(d[COL_HUMAN], errors="coerce")
        full[lang] = d
        work[lang] = d.dropna(subset=["H"]).reset_index(drop=True)
    return full, work


full, work = load_sheets(DATA_XLMR)
print(f"Loaded {sum(len(full[l]) for l in LANG_ORDER):,} rows "
      f"across {len(SHEET_MAP)} sheets")

from scipy import stats
from sklearn.isotonic import IsotonicRegression

A = pd.concat([work[l].assign(lang=l) for l in LANG_ORDER], ignore_index=True)
A["Cn"]  = A[COL_COMET_NAT]
A["Cr"]  = A[COL_COMET_ROM]
A["TPn"] = A[COL_TP_NAT]
A["TPr"] = A[COL_TP_ROM]
A["IPn"] = A[COL_IP_NAT]
A["IPr"] = A[COL_IP_ROM]
A["dTP"] = A["TPn"] - A["TPr"]      # native minus romanised parity delta
A["dIP"] = A["IPn"] - A["IPr"]
print(f"Working set assembled: N = {len(A):,}")

## Step 2 — The 50/50 Within-Language Split

Two details of this split determine the draw, and both are fixed here.

1. The frame is concatenated in **alphabetical** language order (Gujarati,
   Hindi, Malayalam, Marathi, Tamil), so the row indices differ from the working
   set used elsewhere in the pipeline.
2. A single `RandomState(42)` is consumed **one language at a time in that same
   order**, so the iteration order is part of the specification.

This matches `scripts/reproduce_all.py`.

In [ ]:
ALPHA = ["Gujarati", "Hindi", "Malayalam", "Marathi", "Tamil"]
ALPHA_SHEET = {"Gujarati": SHEET_MAP["GUJ"], "Hindi": SHEET_MAP["HIN"],
               "Malayalam": SHEET_MAP["MAL"], "Marathi": SHEET_MAP["MAR"],
               "Tamil": SHEET_MAP["TAM"]}
NUM = [COL_COMET_NAT, COL_COMET_ROM, COL_HUMAN,
       COL_TP_NAT, COL_TP_ROM, COL_IP_NAT, COL_IP_ROM]

frames = []
for name in ALPHA:
    f = pd.read_excel(DATA_XLMR, sheet_name=ALPHA_SHEET[name])
    f = f.loc[:, ~f.columns.str.startswith("Unnamed")]
    for c in NUM:
        f[c] = pd.to_numeric(f[c], errors="coerce")
    f = f.dropna(subset=NUM).copy()
    f["Language"] = name
    frames.append(f[NUM + ["Language"]])

D = pd.concat(frames, ignore_index=True)
D["Cn"], D["Cr"], D["H"] = D[COL_COMET_NAT], D[COL_COMET_ROM], D[COL_HUMAN]

rng = np.random.RandomState(SEED_SPLIT)
D["fold"] = 0
for name in ALPHA:
    idx = D.index[D.Language == name].to_numpy()
    test = rng.choice(idx, size=len(idx) // 2, replace=False)
    D.loc[test, "fold"] = 1

TR, TE = D[D.fold == 0], D[D.fold == 1]
print(f"train = {len(TR):,}   test = {len(TE):,}")
for name in ALPHA:
    print(f"  {name:10s} train {len(TR[TR.Language == name]):>4}  "
          f"test {len(TE[TE.Language == name]):>4}")

## Step 3 — The Correctors

Each returns corrected romanised scores for the held-out half of one language,
using only parameters fitted on that language's training half.

In [ ]:
def qn_map(scores, reference_sorted):
    """Rank -> fractional rank -> empirical quantile of the reference."""
    frac = stats.rankdata(scores) / (len(scores) + 1)
    return np.quantile(reference_sorted, frac)


def c_raw(lang):
    """Identity: the uncorrected romanised score."""
    return TE[TE.Language == lang]["Cr"].values


def c_mean_shift(lang):
    """Location only: add the train-half native-minus-romanised mean offset."""
    tr, te = TR[TR.Language == lang], TE[TE.Language == lang]
    return te["Cr"].values + (tr["Cn"].mean() - tr["Cr"].mean())


def c_affine(lang):
    """Location and scale: match the train-half native mean and SD."""
    tr, te = TR[TR.Language == lang], TE[TE.Language == lang]
    return (te["Cr"].values - tr["Cr"].mean()) / tr["Cr"].std() * tr["Cn"].std() + tr["Cn"].mean()


def c_quantile(lang):
    """Full distribution match onto the train-half native quantiles."""
    tr, te = TR[TR.Language == lang], TE[TE.Language == lang]
    r = np.searchsorted(np.sort(tr["Cr"].values), te["Cr"].values) / len(tr)
    r = np.clip(r, 0.001, 0.999)
    return np.quantile(tr["Cn"].values, r)


def c_isotonic(lang):
    """Paired monotone regression, romanised -> native, fitted on the train half."""
    tr, te = TR[TR.Language == lang], TE[TE.Language == lang]
    model = IsotonicRegression(out_of_bounds="clip").fit(tr["Cr"], tr["Cn"])
    return model.predict(te["Cr"].values)


def c_native(lang):
    """The ceiling: the native score itself."""
    return TE[TE.Language == lang]["Cn"].values


print("Six correctors defined (five monotone, plus the native ceiling).")

## Step 4 — Table 5

Three columns: mean absolute per-language gap against the native mean,
Spearman ρ averaged over languages, and pooled Spearman ρ across all ten cells.

In [ ]:
def evaluate(fn):
    gaps, rho_lang, pooled_x, pooled_h = [], [], [], []
    for lang in ALPHA:
        te = TE[TE.Language == lang]
        x = np.asarray(fn(lang), dtype=float)
        gaps.append(abs(np.mean(x) - te["Cn"].mean()))
        rho_lang.append(stats.spearmanr(x, te["H"])[0])
        pooled_x += list(x)
        pooled_h += list(te["H"])
    return np.mean(gaps), np.mean(rho_lang), stats.spearmanr(pooled_x, pooled_h)[0]


CORRECTORS = [
    ("Raw romanised (identity)",         "raw",            c_raw),
    ("Per-lang mean shift (location)",   "mean_shift",     c_mean_shift),
    ("Per-lang affine / z (loc.+scale)", "affine",         c_affine),
    ("Per-lang quantile map (monotone)", "quantile",       c_quantile),
    ("Per-lang isotonic (monotone)",     "isotonic",       c_isotonic),
    ("Native score (ceiling)",           "native_ceiling", c_native),
]

rows = []
print("Table 5 — the impossibility result (50/50 within-language split, seed 42)")
print(f"{'Corrector':36s}  {'|gap|':>7}  {'\u03c1_lang':>8}  {'\u03c1_pool':>8}")
print("-" * 64)
for label, key, fn in CORRECTORS:
    gap, rl, rp = evaluate(fn)
    rows.append(dict(corrector=key, label=label, gap=gap, rho_lang=rl, rho_pool=rp))
    print(f"{label:36s}  {gap:>7.2f}  {rl:>8.3f}  {rp:>8.3f}")

table7 = pd.DataFrame(rows).set_index("corrector")

## Step 5 — Cross-Verify the Impossibility Claim

In [ ]:
monotone = table7.drop(index="native_ceiling")
spread = monotone["rho_lang"].max() - monotone["rho_lang"].min()
ceiling = table7.loc["native_ceiling", "rho_lang"]
raw_gap = table7.loc["raw", "gap"]

# Every monotone corrector must leave within-language ranking untouched.
assert spread < 0.005, f"\u03c1_lang varies by {spread:.4f} across monotone correctors"
# Yet all of them close the gap.
assert (monotone.drop(index="raw")["gap"] < 1.0).all()
# And none of them approaches the native ceiling.
assert ceiling - monotone["rho_lang"].max() > 0.2

# ── Cross-verification against Table 5 ───────────────────────────────────────
assert abs(raw_gap - 7.88) < 0.005, f"raw gap = {raw_gap:.2f}, Table 5 says 7.88"
assert abs(table7.loc["raw", "rho_lang"] - 0.310) < 0.001
assert abs(table7.loc["raw", "rho_pool"] - 0.160) < 0.001
assert abs(ceiling - 0.564) < 0.001, f"ceiling = {ceiling:.3f}, Table 5 says 0.564"
print(f"\u2713 Raw gap = {raw_gap:.2f} pts (Table 5: 7.88)")

print(f"\u2713 \u03c1_lang is frozen at \u2248 {monotone['rho_lang'].mean():.3f} across all five "
      f"monotone correctors (spread {spread:.4f})")
print(f"\u2713 Gap closes from {raw_gap:.2f} to "
      f"< {monotone.drop(index='raw')['gap'].max():.2f} pts")
print(f"\u2713 Native ceiling \u03c1_lang = {ceiling:.3f} — no corrector gets near it "
      f"(best: {monotone['rho_lang'].max():.3f})")
print("\n  Monotone correction buys comparability of location, never of ranking.")

## Step 6 — Save

In [ ]:
path = TABLES_DIR / "table5_corrector_impossibility.csv"
table7.to_csv(path)
print(table7.round(3).to_string())
print(f"\nSaved \u2192 {path}")

## Step 7 — Output Manifest

In [ ]:
print("=== Notebook 05 — output manifest ===")
print("  table5_corrector_impossibility.csv")

## References

**This work.**
Anonymous (2026). *Under review.*

**Information Parity (IP).**
Tsvetkov, A., & Kipnis, A. (2024). Information Parity: Measuring and Predicting the
Multilingual Capabilities of Language Models. *Findings of EMNLP 2024*, pp. 7971–7989.

**Tokenization Parity and tokenizer unfairness.**
Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language Model Tokenizers
Introduce Unfairness Between Languages. *NeurIPS 36*.

**COMET.**
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for
MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213

**IndicMT Eval dataset.**
Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., &
Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics
for Indian Languages. *ACL 2023*, pp. 14210–14228. https://aclanthology.org/2023.acl-long.795

**WMT24 Latin-script controls.**
Kocmi, T., et al. (2024). Findings of the WMT24 General Machine Translation Shared Task.
*Proceedings of WMT 2024*, pp. 1–46. https://aclanthology.org/2024.wmt-1.1